# Исследование гиперпараметров генетического алгоритма

**Задача:** задача о назначениях; точный эталон — `scipy.optimize.linear_sum_assignment`.

Три условных класса: **малые** 3×3/5×5, **средние** 8×8/12×12, **большие** 16×16/24×24. На каждой размерности создаются одинаковые случайные матрицы для всех сравниваемых настроек; варьируется только **один гиперпараметр**.

Измеряются среднее относительное отклонение от оптимума, доля точных решений, число поколений, время выполнения и разброс. Время включает вызов SciPy для эталона.

In [ ]:
from pathlib import Path
import csv, sys
from IPython.display import display, Markdown, Image

lab_dir = Path.cwd() if (Path.cwd() / 'experiments.py').exists() else Path.cwd() / 'lab1'
sys.path.insert(0, str(lab_dir.resolve()))
from experiments import SWEEPS, BASELINE, SIZE_CLASSES, save_study
root = lab_dir.parent
results = root / 'results'
if not (results / 'summary.csv').exists():
    save_study(results, matrix_count=6, repeats=2)
with (results / 'summary.csv').open(encoding='utf-8') as f:
    summary = list(csv.DictReader(f))
print('Baseline:', BASELINE)
print('Classes:', SIZE_CLASSES)
print('Summary rows:', len(summary))


## Результаты однофакторных экспериментов

Каждая кривая относится к отдельному классу размерности. Значение Y на графике отклонения выражено в процентах. Усредняются все матрицы и повторы класса. Доля точных решений и время выполнения содержатся в отдельных графиках каталога `results/plots`.


In [ ]:
for parameter in SWEEPS:
    display(Markdown(f'### {parameter}'))
    display(Image(filename=str(results / 'plots' / f'{parameter}_mean_gap.png')))
    items = [r for r in summary if r['parameter'] == parameter and r['size'] == 'all']
    display(Markdown('| Класс | Значение | Отклонение, % | Точных решений, % | Среднее время, с |\n|---|---:|---:|---:|---:|\n' + '\n'.join(f"| {r['class']} | {r['value']} | {100*float(r['mean_gap']):.2f} | {100*float(r['optimal_rate']):.1f} | {float(r['mean_elapsed_seconds']):.3f} |" for r in items)))


## Ограничения и выводы

Изменение одной настройки при одинаковых матрицах и seed позволяет интерпретировать различия между кривыми как влияние параметра при данном экспериментальном протоколе. Небольшие различия между средними не доказывают отсутствие влияния без статистической оценки. Размерности 3–24 — условные классы внутри данного эксперимента, а не универсальная классификация сложности. Параллельное выполнение островов в этот эксперимент не входит.
